In [3]:
# ==============================
# Import Libraries
# ==============================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

import joblib

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [4]:
# ==============================
# Load Dataset
# ==============================

df = pd.read_csv("../data/Synthetic_Dataset.csv")

print("Dataset Loaded Successfully")

df.head()

Dataset Loaded Successfully


,Source,Destination,Source_Type,Dest_Type,Source_Lat,Source_Lon,Dest_Lat,Dest_Lon,Road_Distance_km,Road_Type,...,Theft_Reports,Harassment_Reports,Police_Distance_km,CCTV,Emergency_Booth,Shops_Open,Public_Transport,Road_Condition,Safety_Score,Safety_Level
0,Kothrud,Karve Nagar,Residential,Residential,18.5074,73.8077,18.4898,73.8145,2.09,Street,...,11,7,3.13,Yes,Yes,Yes,Yes,Average,83.5,Safe
1,Fatima Nagar,VIT Pune,Residential,Educational,18.4968,73.9010,18.4635,73.8682,5.19,Street,...,21,13,3.73,Yes,No,No,No,Average,52.8,Moderate
2,FC Road,JM Road,Commercial,Commercial,18.5204,73.8417,18.5232,73.8445,0.44,Main Road,...,2,1,2.09,Yes,No,Yes,Yes,Average,100.0,Safe
3,Baner,Aundh,Residential,Residential,18.5590,73.7868,18.5600,73.8070,2.24,Service Road,...,2,2,0.45,No,Yes,No,No,Good,100.0,Safe
4,Lohegaon,Koregaon Park,Residential,Commercial,18.5814,73.9197,18.5362,73.8930,5.83,Service Road,...,25,10,4.99,No,Yes,Yes,No,Good,23.0,Unsafe


In [5]:
# ==============================
# Duplicate Rows
# ==============================

print("Duplicate Rows :", df.duplicated().sum())

df = df.drop_duplicates()

print("Shape After Removing Duplicates :", df.shape)

Duplicate Rows : 0
Shape After Removing Duplicates : (30000, 32)


In [17]:
# ==============================
# Features and Target
# ==============================

X = df.drop(columns=[
    "Safety_Level",
    "Safety_Score"
], errors="ignore")

y = df["Safety_Level"]

print(X.shape)
print(y.shape)

(30000, 30)
(30000,)


In [7]:
# ==============================
# Convert Boolean Columns
# ==============================

bool_cols = X.select_dtypes(include="bool").columns

X[bool_cols] = X[bool_cols].astype(int)

print("Boolean Columns Converted")

Boolean Columns Converted


In [8]:
# ==============================
# Numerical & Categorical Columns
# ==============================

categorical_cols = X.select_dtypes(include="object").columns

numerical_cols = X.select_dtypes(exclude="object").columns

print("Categorical Columns")
print(categorical_cols)

print()

print("Numerical Columns")
print(numerical_cols)

Categorical Columns
Index(['Source', 'Destination', 'Source_Type', 'Dest_Type', 'Road_Type',
       'Season', 'Weather', 'Street_Lighting', 'Visibility', 'Crowd_Density',
       'Vehicle_Traffic', 'Pedestrian_Traffic', 'Crime_Rate', 'CCTV',
       'Emergency_Booth', 'Shops_Open', 'Public_Transport', 'Road_Condition'],
      dtype='object')

Numerical Columns
Index(['Source_Lat', 'Source_Lon', 'Dest_Lat', 'Dest_Lon', 'Road_Distance_km',
       'Is_Weekend', 'Is_Festival', 'Hour', 'Crime_Probability',
       'Theft_Reports', 'Harassment_Reports', 'Police_Distance_km'],
      dtype='object')


In [9]:
# ==============================
# Label Encode Target
# ==============================

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(y)

print("Classes")

for i, label in enumerate(label_encoder.classes_):
    print(i, "->", label)

Classes
0 -> Moderate
1 -> Safe
2 -> Unsafe


In [10]:
# ==============================
# One Hot Encoding
# ==============================

X = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print("Shape After Encoding")

print(X.shape)

Shape After Encoding
(30000, 113)


In [11]:
# ==============================
# Feature Scaling
# ==============================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

X_scaled = pd.DataFrame(
    X_scaled,
    columns=X.columns
)

X_scaled.head()

,Source_Lat,Source_Lon,Dest_Lat,Dest_Lon,Road_Distance_km,Is_Weekend,Is_Festival,Hour,Crime_Probability,Theft_Reports,...,Pedestrian_Traffic_Low,Pedestrian_Traffic_Medium,Crime_Rate_Low,Crime_Rate_Medium,CCTV_Yes,Emergency_Booth_Yes,Shops_Open_Yes,Public_Transport_Yes,Road_Condition_Good,Road_Condition_Poor
0,-0.630286,-0.576417,-0.993139,-0.438821,-0.729245,1.605160,-0.227558,1.367423,0.915664,0.753514,...,1.243833,-0.656525,-0.898702,1.108300,0.474225,0.730362,0.867609,0.479018,-0.902891,-0.421237
1,-0.843370,1.136124,-1.498239,0.553295,-0.048593,-0.622991,-0.227558,-1.380565,1.565124,2.512977,...,1.243833,-0.656525,-0.898702,-0.902283,0.474225,-1.369184,-1.152592,-2.087606,-0.902891,-0.421237
2,-0.368957,0.047661,-0.351681,0.115434,-1.091528,-0.622991,-0.227558,-0.657410,-1.422393,-0.830003,...,-0.803966,-0.656525,1.112716,-0.902283,0.474225,-1.369184,0.867609,0.479018,-0.902891,-0.421237
3,0.406990,-0.960040,0.355076,-0.577385,-0.696311,-0.622991,-0.227558,1.222792,-0.729635,-0.830003,...,-0.803966,-0.656525,1.112716,-0.902283,-2.108702,0.730362,-1.152592,-2.087606,1.107554,-0.421237
4,0.857280,1.479367,-0.102011,1.011479,0.091929,1.605160,-0.227558,1.656685,2.604260,3.216762,...,1.243833,-0.656525,-0.898702,-0.902283,-2.108702,0.730362,0.867609,-2.087606,1.107554,-0.421237


In [12]:
# ==============================
# Train Test Split
# ==============================

X_train, X_test, y_train, y_test = train_test_split(

    X_scaled,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training Shape")

print(X_train.shape)

print()

print("Testing Shape")

print(X_test.shape)

Training Shape
(24000, 113)

Testing Shape
(6000, 113)


In [14]:
# ==============================
# Save Scaler & Label Encoder
# ==============================

joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

joblib.dump(
    label_encoder,
    "../models/label_encoder.pkl"
)

print("Saved Successfully")

Saved Successfully


In [15]:
# ==============================
# Save Processed Dataset
# ==============================

X_train.to_csv(
    "../data/X_train.csv",
    index=False
)

X_test.to_csv(
    "../data/X_test.csv",
    index=False
)

pd.DataFrame(y_train).to_csv(
    "../data/y_train.csv",
    index=False
)

pd.DataFrame(y_test).to_csv(
    "../data/y_test.csv",
    index=False
)

print("Processed Dataset Saved")

Processed Dataset Saved


In [16]:
print("="*50)

print("Final Dataset")

print("="*50)

print("Training Samples :", len(X_train))

print("Testing Samples :", len(X_test))

print()

print("Features :", X_train.shape[1])

print()

print("Classes :", label_encoder.classes_)

Final Dataset
Training Samples : 24000
Testing Samples : 6000

Features : 113

Classes : ['Moderate' 'Safe' 'Unsafe']
